In [1]:
import requests
import pandas as pd
from io import StringIO

SEASON_URL = "https://sportsbookreviewsonline.com/scoresoddsarchives/nfl-odds-2021-22"

## Functions

In [2]:
def fetch_season_table(url: str) -> pd.DataFrame:
    """fetch the page and pull out the raw odds table."""
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    tables = pd.read_html(StringIO(resp.text))
    # the odds table is the largest table on the page
    raw = max(tables, key=len)

    # site renders the header as a normal first data row (no <th>), so
    # pandas assigns 0..N as column names -- promote row 0 to headers.
    if list(raw.columns) == list(range(raw.shape[1])) or all(
        str(c).isdigit() for c in raw.columns
    ):
        raw.columns = [str(c).strip() for c in raw.iloc[0]]
        raw = raw.iloc[1:].reset_index(drop=True)
    else:
        raw.columns = [str(c).strip() for c in raw.columns]

    return raw


def _to_num(x):
    """Convert 'pk' (pick'em) to 0.0, blanks to NaN, else float."""
    if pd.isna(x):
        return float("nan")
    x = str(x).strip().lower()
    if x in ("", "nl"):
        return float("nan")
    if x == "pk":
        return 0.0
    try:
        return float(x)
    except ValueError:
        return float("nan")


def parse_games(raw: pd.DataFrame) -> pd.DataFrame:
    """pair away/home rows into one game per row, and split the
    dual-purpose Open/Close columns into spread_line vs total_line."""
    df = raw.copy()

    rename_map = {}
    for c in df.columns:
        cl = c.lower()
        if cl == "date":
            rename_map[c] = "date"
        elif cl == "rot":
            rename_map[c] = "rot"
        elif cl == "vh":
            rename_map[c] = "vh"
        elif cl == "team":
            rename_map[c] = "team"
        elif cl == "final":
            rename_map[c] = "final"
        elif cl == "open":
            rename_map[c] = "open"
        elif cl == "close":
            rename_map[c] = "close"
        elif cl == "ml":
            rename_map[c] = "ml"
        elif cl == "2h":
            rename_map[c] = "second_half"
    df = df.rename(columns=rename_map)

    keep = ["date", "rot", "vh", "team", "final", "open", "close", "ml"]
    df = df[[c for c in keep if c in df.columns]].copy()

    df["rot"] = pd.to_numeric(df["rot"], errors="coerce")
    df["final"] = pd.to_numeric(df["final"], errors="coerce")
    df["open_num"] = df["open"].apply(_to_num)
    df["close_num"] = df["close"].apply(_to_num)
    df["ml_num"] = pd.to_numeric(df["ml"], errors="coerce")

    # rows already come pre-interleaved (away, home, away, home, ...) in
    # the source table -- do NOT sort by rot globally, since rotation
    # numbers reset every week and a global sort scrambles the pairing.
    df = df.dropna(subset=["rot"]).reset_index(drop=True)

    games = []
    i = 0
    while i < len(df) - 1:
        away = df.iloc[i]
        home = df.iloc[i + 1]

        # basic sanity check: rotation numbers should be consecutive (paired)
        if int(home["rot"]) - int(away["rot"]) != 1:
            i += 1
            continue

        # disambiguate spread vs total: the total is the larger magnitude
        # number and is (usually) identical/near-identical on both rows;
        # the spread is the smaller number and differs between rows.
        def split_spread_total(a_val, h_val):
            pair = [a_val, h_val]
            total = max(pair, key=lambda v: abs(v) if pd.notna(v) else -1)
            spread = min(pair, key=lambda v: abs(v) if pd.notna(v) else float("inf"))
            return spread, total

        open_spread, open_total = split_spread_total(away["open_num"], home["open_num"])
        close_spread, close_total = split_spread_total(away["close_num"], home["close_num"])

        games.append({
            "date": away["date"],
            "away_team": away["team"],
            "home_team": home["team"],
            "away_score": away["final"],
            "home_score": home["final"],
            "away_ml": away["ml_num"],
            "home_ml": home["ml_num"],
            "open_spread": open_spread,
            "close_spread": close_spread,
            "open_total": open_total,
            "close_total": close_total,
        })
        i += 2

    return pd.DataFrame(games)

## Pulling data

In [3]:
raw = fetch_season_table(SEASON_URL)
games = parse_games(raw)
games.shape

(285, 11)

In [4]:
games.head(10)

,date,away_team,home_team,away_score,home_score,away_ml,home_ml,open_spread,close_spread,open_total,close_total
0,909,Dallas,TampaBay,29,31,375,-450,7.0,10.0,52.5,52.5
1,912,Pittsburgh,Buffalo,23,16,240,-280,7.0,7.0,51.0,47.5
2,912,NYJets,Carolina,14,19,160,-180,4.0,3.5,43.5,44.5
3,912,Jacksonville,Houston,21,37,-170,150,0.0,3.0,46.0,45.5
4,912,Arizona,Tennessee,38,13,125,-145,2.5,2.5,51.5,54.0
5,912,LAChargers,Washington,20,16,110,-130,0.0,1.0,45.0,45.5
6,912,Philadelphia,Atlanta,32,6,155,-175,3.5,3.5,47.0,48.5
7,912,Seattle,Indianapolis,28,16,-145,125,2.5,2.5,52.5,49.0
8,912,Minnesota,Cincinnati,24,27,-155,135,3.0,3.0,48.5,47.5
9,912,SanFrancisco,Detroit,41,33,-430,360,9.0,10.0,46.0,46.0
